# Wie funktioniert .query mit Aufruf von set._res.query?

# Python packages installation

A good [introduction](https://github.com/astrofrog/py4sci) to Python in [Jupyter Notebooks](http://jupyter.kip.uni-heidelberg.de/) is recommended for students that are new to using Python for data acquisition and analysis.

We are using the [JupyterLab](https://jupyter.org/) server with the [SciPy Notebook](https://hub.docker.com/r/jupyter/scipy-notebook) that is available for free and can be used on any computer.

First we install all additional Python packages required for this experiment with pip.

In [1]:
!pip3 install pyvisa-py pyserial pyusb zeroconf tqdm

In [2]:
pip install uncertainties

Note: you may need to restart the kernel to use updated packages.


# Communication with hardware


**Important:** All devices have to be enabled _prior_ to starting the JupyterLab server. If communication with hardware fails, shut down the server, enable _all_ devices, then start the JupyterLab server again.

## **PyVISA** library

**V**irtual **I**nstrument **S**oftware **A**rchitecture ([VISA](https://en.wikipedia.org/wiki/Virtual_instrument_software_architecture)) is a widely used application programming interface (API) in the test and measurement (T&M) industry for communicating with instruments from a computer.

* [PyVISA](https://pyvisa.readthedocs.io/en/latest/) is a Python package that allows to control all kinds of measurement devices independently of the interface (e.g. GPIB, RS232, USB, Ethernet).

* [PyVISA-py](https://pyvisa.readthedocs.io/projects/pyvisa-py/en/latest/) is a backend for **PyVISA**. It implements most of the methods for Message Based communication (Serial, USB, GPIB, Ethernet) using Python and some well developed, easy to deploy and cross platform libraries.

After importing the **PyVISA** library via ``import pyvisa`` the **PyVISA-py** backend can be used by choosing ``@py`` when instantiating the visa Resource Manager. 

Invoking the ``list_resources()`` method provides all locally available resources. 

## **USB** adapters for communication

The adapters have to be attached to the computer _and_ the devices enabled _before_ starting the JupyterLab Server, because the kernel module ``ftdi_sio`` will be reloaded with ``modprobe -r ftdi_sio ; modprobe ftdi_sio`` on starting the server and then probe the devices and configure their communication settings according to the devices attached. If communication does not work (due to devices not attached or enabled prior to starting the server) restarting the JupyterLab Server (closing the terminal and starting it again) may fix the issue.

In Linux the devices with USB ID ``0403:6001`` will be used by the kernel driver ``ftdi_sio`` that creates device files at ``/dev/ttyUSB0`` ...  ``/dev/ttyUSB1`` etc. with the lowest free name available. 

When there are multiple devices, the udev system allows to distinguish (e.g. via attributes in ``udevadm info --attribute-walk --name=ttyUSB0``) and create a stable mapping with udev rules in ``/etc/udev/rules.d/`` to ensure that unique symlinks always point to the same devices, even in case they happen to end up with other device files:

<!---
 * The [US232R-100-BULK](https://ftdichip.com/wp-content/uploads/2023/07/DS_US232R-10_R-100-500.pdf) is a generic USB to serial (RS232) adapter manufactured by [FTDI](https://ftdichip.com/). The file ``99-ftdi.rules`` creates ``/dev/ttyUSBSerial`` with ``SUBSYSTEMS=="usb", ATTRS{idVendor}=="0403", ATTRS{idProduct}=="6001", ATTRS{manufacturer}=="FTDI", ATTRS{product}=="US232R", MODE="0666", SYMLINK+="ttyUSBSerial"`` which will be listed as ``ASRL/dev/ttyUSBSerial::INSTR`` by **PyVISA**. 
-->

* The [Prologix GPIB-USB controller](https://prologix.biz/downloads/PrologixGpibUsbManual-6.0.pdf) is a USB bridge to GPIB (IEEE-488) adapter manufactured by [Prologix ](https://prologix.biz/). The file ``99-prologix.rules`` creates ``/dev/ttyUSBGPIB`` with ``SUBSYSTEMS=="usb", ATTRS{idVendor}=="0403", ATTRS{idProduct}=="6001", ATTRS{manufacturer}=="Prologix", ATTRS{product}=="Prologix GPIB-USB Controller", MODE="0666", SYMLINK+="ttyUSBGPIB"`` which will be listed as ``ASRL/dev/ttyUSBSerial::INSTR`` by **PyVISA**.

In this setup we use the Prologix GPIB-USB controller as an USB bridge to GPIB devices such as the Synthesized Function Generator (GPIB Address 19) or the Lock-In Amplifier (GPIB Address 7).

<!--
In addition we can use the generic FTDI USB to serial adapter to communicate with RS232 ports, in this case we need to ensure correct crossover mapping between receive (RXD) and transmit (TXD) pins of the RS232 cable (RXD@adapter pin 2 connected with TXD@device pin 3) and (TXD@adapter pin 3 connected with RXD@device pin 2), e.g. a "cross-over" cable with two female 9-pin RS232 connectors.
-->


## **Prologix GPIB-USB** controller

The [Prologix GPIB-USB controller](https://prologix.biz/downloads/PrologixGpibUsbManual-6.0.pdf) converts any computer with a USB port into a **G**eneral **P**urpose **I**nterface **B**us ([GPIB](https://en.wikipedia.org/wiki/IEEE-488)) Controller or Device.

We use the Prologix GPIB-USB controller as an USB bridge to GPIB, it apppears as [FTDI](https://ftdichip.com/) FT232 (UART IC USB bridge with USB ID ``0403:6001``) as serial device ``/dev/ttyUSBGPIB`` in Linux and can be accessed as ``ASRL/dev/ttyUSBGPIB::INSTR`` in **PyVISA**.

Querying the resource with ``++ver`` asks the controller for its version, it will respond with its firmware version string.

In [3]:
import pyvisa # hardware access

rm  = pyvisa.ResourceManager('@py')                  # choosing the PyVISA-py backend
res = rm.open_resource('ASRL/dev/ttyUSBGPIB::INSTR') # open Prologix GPIB-USB controller
print(res.query("++ver"))                            # Query Prologix GPIB-USB controller for its firmware version

Prologix GPIB-USB Controller version 6.107



The Prologix GPIB-USB controller interprets high level commands received from the host computer and performs the appropriate low-level GPIB protocol
handshaking.

* In **Device mode** (``++mode 0``), Prologix GPIB-USB controller converts the computer into a GPIB peripheral for downloading data and screen plots from the instrument front panel. 

* In **Controller mode** (``++mode 1``), Prologix GPIB-USB controller can remotely control GPIB enabled instruments such as Oscilloscopes, Logic Analyzers, and Spectrum Analyzers.

In our case we will use the Prologix GPIB-USB controller in controller mode with ``++mode 1`` and choose to talk to the GPIB device at address 19 via ``++addr 19``.

All commands to the controller have a prefix of ``++`` while the everything else will be written on GPIB to the device address. 

The chars ESC (``x1b``), CR (``x0d``), LF (``x0a``) and PLUS (``x1b``) have to be escaped by an ESC (``x1b``) prefix.

In [ ]:
rm  = pyvisa.ResourceManager('@py')                  # Choose the PyVISA-py backend
res = rm.open_resource('ASRL/dev/ttyUSBGPIB::INSTR') # Open Prologix GPIB-USB controller
res.write("++mode 1")                                # Put Prologix GPIB-USB in Controller-In-Charge (CIC) mode on the GPIB
res.write("++addr 19")                               # Choose device on GPIB address 19 for further communication
res.write("FREQ 123.456")                            # Write a command to this device on GPIB

14

## **Python classes**

We will use Python [classes](https://docs.python.org/3/tutorial/classes.html) to implement functionality in a hierarchy of sections that naturally divide a task into smaller sub-tasks that can be independently implemented and verified. 

Once a sub-task is finished the next sub-task can inherit the methods and attributes of the previous sub-task and this hierarchy can be expressed in individual Python classes.

The first argument in each method of a Python class is ```self``` referring to this instance of the Python class and can be used to to address attributes and methods of this instance.

In [ ]:
class Base():                                                    # Class name
    def __init__(self, name="Default", purpose="communication"): # Constructor gets called when instance of class is created
        self.name = name                                         # Attribute of instance "self" called "name" will be set to value of variable "name"
        self.purpose = purpose                                   # Attribute set
        self._setText("Class {name} for {purpose}.")             # Call method on this instance "self"

    def _setText(self, text):                                    # Method definition, first argument always is "self" refering to this instance
        self._text = text                                        # Attribute set, underscore prefix indicates internal usage

    def help(self):                                              # Method definition, first argument always is "self" refering to this instance
        print(self._text.format(name=self.name, purpose=self.purpose))

base = Base("GPIB", "hardware communication")                    # Instance of class created and constructor called with arguments. Inside the class the self reference is used for this
base.help()  

Class GPIB for hardware communication.


## **GPIB** class for underlying communication commands to GPIB

A generic Python class for GPIB devices via the [Prologix GPIB-USB controller](https://prologix.biz/downloads/PrologixGpibUsbManual-6.0.pdf).

In [ ]:
import pyvisa,re

class GPIB(Base): # Create class GPIB that interits the methods and attributes of the Base class
    """Use GPIB devices via Prologix GPIB-USB Controller Rev 6.4.1"""
    # https://pyvisa.readthedocs.io/en/latest/
    # https://prologix.biz/downloads/PrologixGpibUsbManual-6.0.pdf

    def __init__(self, dev="ttyUSBGPIB", addr=None, res=None):   # Constructor
        self.__parent = super(GPIB, self)                        # https://docs.python.org/3/library/functions.html#super
        self.__parent.__init__("GPIB", "hardware communication") # call constructor of parent class with arguments      
        self._rm   = pyvisa.ResourceManager("@py")               # using PyVISA-py
        self._dev  = self.setDev(dev)   if dev  else None
        self._addr = self.setAddr(addr) if addr else None
        self._res  = self.setRes(res)   if res  else self._res
        self._esc  = ['\x1b', '\x0d', '\x0a', '\x2b']

    def setDev(self, dev=None):
        if dev:
            self._dev = str(dev)
        if self._rm and self._dev:
            for res in sorted(self._rm.list_resources()):
                if self._dev in res:
                    self._res = self._rm.open_resource(res)
                    self.setRes(self._res)
        return self._dev
    
    def setRes(self, res=None):
        if res:
            self._res = res
        if self._res:
            self._res.read_termination  = "\n"
            self._res.write_termination = "\n"
            self._res.write("++mode 1") # "++mode" is "0" (GPIB TALKER or GPIB LISTENER) or "1" (Controller-In-Charge (CIC))
        return self._res        

    def setAddr(self, addr=None):
        if addr:
            self._addr = int(addr)
        if self._res and self._addr:
            self._res.write("++addr " + str(self._addr))
        return self._addr
                
    def _escape(self, data):
        # xlat_t = { ord(c):'\x1b' + c for c in '\x1b\r\n+' }; '+A\rB\nC\x1b'.translate(xlat_t) #==> '\x1b+A\x1b\rB\x1b\nC\x1b\x1b'
        for b in ['\x1b', '\x0d', '\x0a', '\x2b']: # ESC, CR, LF, PLUS (ESC MUST BE FIRST)
            data = data.replace(b, '\x1b' + b) # prepend with ESC
        return data
            
    def write(self, addr, data):
        if addr and data:
            self.setAddr(addr)
            if self._res:
                return self._res.write(self._escape(data))

    def query(self, addr, data, dtype="float"):
        if addr and data and dtype:
            self.setAddr(addr)
            if self._res:
                if dtype == "float":
                    return float(re.sub('[^0-9eE\.+-]', '',self._res.query(self._escape(data))))
                else:
                    return self._res.query(data)

### Below we demonstrate an example usage of an instance from this GPIB() class:

### Prologix GPIB-USB controller
gpib = GPIB("ttyUSBGPIB") # create an instance from the GPIB() class on the device /dev/ttyUSBGPIB
print(gpib._res.query("++ver"))

### call our help method from the parent class
gpib.help()

### SRS SD345 30 MHz Synthesized Function Generator on GPIB address 19
freq = 1234.567 # Frequency in Hz <= 30E6
gpib.write(19, "FREQ {:.3f}".format(freq))
freq = gpib.query(19, "FREQ?")
print("freq = {:.3f} Hz".format(freq))

ampl = 10.00 # Amplitude in Vpp <= 10
gpib.write(19, "AMPL {0:.2f}VP".format(ampl))
ampl = gpib.query(19, "AMPL?")
print("ampl = {0:.2f} Vpp".format(ampl))

### SRS SR530 Lock-In Amplifier on GPIB address 7
qx = gpib.query(7, "QX") # X in V
qy = gpib.query(7, "QY") # Y in V
print("(x,y) = ({x:.2E} V, {y:.2E} V)". format(x=qx, y=qy))

inp_key = "X1" # or "X2" according to hw setup label on lock-in amp
inp_val = gpib.query(7, inp_key) # input in Vdc
print("input = {0:.3f} V". format(inp_val))

out_key = "X5" # or "X6" according to hw setup label on lock-in amp
out_val = 0.512 # output in V e.g. -0.123 or 0.456
gpib.write(7, "{key},{val:.3f}".format(key=out_key, val=out_val))
out_val = gpib.query(7, out_key)
print("output = {0:.3f} V". format(out_val))
gpib.write(7, "{0},0.000".format(out_key)) # reset to 0 (to avoid cooling or heating via peltier element)
print("output reset back to " + str(gpib.query(7, out_key)) + " V")

Prologix GPIB-USB Controller version 6.107
Class GPIB for hardware communication.
freq = 1234.567 Hz
ampl = 10.00 Vpp
(x,y) = (8.88E-05 V, 3.78E-06 V)
input = 1.130 V
output = 0.513 V
output reset back to 0.0 V


## **SynFunGen** class for 30 MHz Synthesized Function Generator SRS DS345

A Python class for communication via GPIB with the 30 MHz Synthesized Function Generator (Stanford Research Systems Model [DS345](https://www.thinksrs.com/products/ds345.html)) inherits from the GPIB class.

In [ ]:
# TODO for students: Implement a class SynFunGen that inherits from GPIB class and controls the function generator
class SFG(GPIB):
    
    def set_f(self,freq):
        self.write(19, "FREQ {:.3f}".format(freq))

    def read_f(self):
        return self.query(19, "FREQ?")

## **LockInAmp** class for Lock-In Amplifier SRS SR530

A Python class for communication via GPIB with the Lock-In Amplifier (Stanford Research Systems [SR530](https://www.thinksrs.com/products/sr510530.htm)) that inherits everything from the SynFunGen class (including its parent GPIB class, hence everything for implementation aswell).

In [ ]:
# TODO for students: Implement a class LockInAmp th2460at inherits from the function generator class and controls the Lock-In amplifier
import numpy as np
import time
from time import sleep
import timeit

import uncertainties as unc
from uncertainties import ufloat
from uncertainties.umath import atan2

class LIAmp(SFG):
    def read(self):
        qx = self.query(7, "QX") # X in V
        qy = self.query(7, "QY") # Y in V
        return qx,qy
        ### SRS SR530 Lock-In Amplifier on GPIB address 7
    

    def amp_phi(self,qx=None,qy=None):
        if qx is None:
            qx,qy1=self.read()
        if qy is None:
            qx1,qy=self.read()
        A=(qx**2+qy**2)**0.5
        phi=atan2(qy,qx) #this returns phi in radians in (-pi,+pi]
        return A,phi 

    def measure_row(self):#arg ,freq
        qx,qy=self.read()
        m_freq=self.read_f()
        row=np.array([m_freq,qx,qy])
        return row

    def wait(self): #waits 0.5 to 1 second
        delay=0.5+np.random.rand()*0.5
        sleep(delay)

    def n_measures(self, number): #  arg freq,  without setting reference freq
        measures=np.empty((number,3))
        for i in range(number):
            temp=self.measure_row()  #arg freq
            measures[i]=temp
            #print(delay," minutes delayed")
            #print(f"{i}")
        return measures

## **DataWriter** class for writing data to files

A Python class for data storage into files. For numerical data using TSV or CSV format is suggested because of its widespread usage and software support.

We recommend the [Python CSV](https://docs.python.org/3/library/csv.html) library that provides a [DictWriter()](https://docs.python.org/3/library/csv.html#csv.DictWriter) method for data storage in CSV format.

In [ ]:
# TODO for students: Implement a class DataWriter that inherits from the Lock-in amp class and
# provides methods to open a file name, write CSV headers and for each measurement a row of columns.
import csv
from pathlib import Path

class DataWriter(LIAmp):
    def __init__(self,path):
        self.pathname=path # gets called when finally creating a Measurement class with argument path='folder' e.g. the WithTemp Measurement
        self._path=Path.cwd()
        LIAmp.__init__(self)    # because DataWriter has its own init class, it needs to manually inherit its parents init aswell
        self.createfile()

    def createfile(self):
        with open(self._path/f"{self.pathname}.csv", 'w', newline='') as csvfile:
            fieldnames = ['FQ','QX',"dQX", 'QY',"dQY", 'A',"dA",'phi',"dphi"]
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            
    def writeData(self,l): # writes basic values QX,QY their statistically calculated errors and the calculated A,dA and Phi,dPhi
        with open(self._path/f"{self.pathname}.csv", 'a', newline='') as csvfile:
            fieldnames = ['FQ','QX',"dQX", 'QY',"dQY", 'A',"dA",'phi',"dphi"]
            writer = csv.DictWriter(csvfile,fieldnames=fieldnames)
            writer.writerow({'FQ':l[0],'QX':l[2],"dQX":l[3], 'QY':l[4],"dQY":l[5], 'A':l[6],"dA":l[7],'phi':l[8],"dphi":l[9]})
            
    def createtemp_file(self): # same as above, but with additional column for current temperature
        with open(self._path/f"{self.pathname}.csv", 'w', newline='') as csvfile:
            fieldnames = ['FQ','QX',"dQX", 'QY',"dQY", 'A',"dA",'phi',"dphi","T_set"]
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            
    def write_tempData(self,l):
        with open(self._path/f"{self.pathname}.csv", 'a', newline='') as csvfile:
            fieldnames = ['FQ','QX',"dQX", 'QY',"dQY", 'A',"dA",'phi',"dphi","T_set"]
            writer = csv.DictWriter(csvfile,fieldnames=fieldnames)
            writer.writerow({'FQ':l[0],'QX':l[2],"dQX":l[3], 'QY':l[4],"dQY":l[5], 'A':l[6],"dA":l[7],'phi':l[8],"dphi":l[9],"T_set":l[10]})


## **Measurement** class for the measurement sequence

Now that we have every functionality ready (and individually tested!) we can finally implement the class that automates the measurement tasks according to the lab course manual.

Using [numpy](https://numpy.org/doc/stable/reference/index.html#reference) we recommend the methods [std()](https://numpy.org/doc/stable/reference/generated/numpy.std.html) and [mean()](https://numpy.org/doc/stable/reference/generated/numpy.mean.html) for calculating the mean value and standard deviation (repeated measurements under the same conditions).

For calculating the phase, we recommend the method [arctan2()](https://numpy.org/doc/stable/reference/generated/numpy.arctan2.html#numpy-arctan2) (described [here](https://en.wikipedia.org/wiki/Atan2) in detail).

In [ ]:
# TODO for students: Implement a class Measurement that inherits from the data writer class (and therefore everything else)
# for fully autonomous measurement sequences and data storage into individual files.

# Flow chart: First: Class creation Example=Measurement(path), then one can call Example.measure_curve(start,end,vals,number) in the final run, where start and end set the boundaries of which frequency section.
# vals is the number of steps between start-end, number decides the amount of values taken for each frequency, default is set to 20, but can be set to a decreased value for test purposes.
# measure_curve itself calls stat_measure, that passes to n_measures -> measure_row: measure_row is the function actually taking the values, which is called 20 times by n_measures to create an array of 20 rows with qx,qy.
# then this array "measures" is passed to list_to_ufloat which takes the mean and std plus calculates A,phi with amp_phi and makes that 8 column array finally saved by writeData

class Measurement(DataWriter):

    def stat_measure(self,number): #arg ,freq  returns mean and error for qx,qy,A,phi
        measures=self.n_measures(number) # arg freq,
        statresults=self.list_to_ufloat(measures)
        return statresults

    def list_to_ufloat(self,list):
            mean=np.mean(list,axis=0)
            sigma=np.std(list,axis=0,ddof=1)
            errors=sigma/(len(list)**0.5)
            f=ufloat(mean[0],errors[1])*2 #this is where we multiply with two again #could use reference frequency here
            qx=ufloat(mean[1],errors[1])
            qy=ufloat(mean[2],errors[2])
            A, phi=self.amp_phi(qx,qy)            
            return np.array([f.n,f.s,qx.n,qx.s,qy.n,qy.s,A.n,A.s,phi.n,phi.s])

    def measure_curve(self,start,end,vals,number=20,log=False): #vals is amount of freqs tested, number is tests for on freq
        if log:
            #check that logspace creates enough values
            values=self.get_log_vals(start,end,vals)
        else:
            values=np.linspace(start,end,vals)
        #wait once:
        for freq in tqdm(values,total=values.shape[0]):
            self.set_f(freq)
            self.wait()
            measures=self.stat_measure(number) #arg freq,
            self.writeData(measures)
           
    def get_log_vals(self,start,end,vals): #gives val amount of numbers in uneven case, val-1 in even case (always returns uneven amount)
        vals=vals+1
        if int(vals*0.5)<2:
            vals=4
        mid=0.5*(start+end)
        valuedister=np.geomspace(1,mid-start+1,int(vals*0.5))
        halfvalue1=mid-valuedister+1
        firstDist=valuedister[1]-valuedister[0]
        halfvalue2=mid+valuedister[1:]-1
        valuess=np.concatenate((halfvalue1,halfvalue2))
        values=np.sort(valuess)
        return values

## Freq Spectrum Intervall Creator
This is used, because one sets the resonant frequencies and then the desired range plus/minus, how far the measurement should measure the peak width.

In [11]:
def interval_abs(mid,radius):
    start=mid-radius
    stop=mid+radius
    return start,stop

def interval_rel(self,mid,delta):
    start=mid*(1-delta*0.01)
    stop=mid*(1+delta*0.01)
    return start, stop

# Temperature communication class
hardware communication with Pt100 and Peltier voltage supply

In [ ]:

class Temp(LIAmp):

    def set_T_U(self,U):
        out_key = "X5" # or "X6" according to hw setup label on lock-in amp
        out_val = U # output in V e.g. -0.123 or 0.456
        self.write(7, "{key},{val:.3f}".format(key=out_key, val=out_val))
        return out_val
    
    def read_T_U(self):
        out_key = "X5" # or "X6" according to hw setup label on lock-in amp
        out_val = self.query(7, out_key)
        return out_val
    
    def read_pT_U(self):
        inp_key = "X1" # or "X2" according to hw setup label on lock-in amp
        inp_val = self.query(7, inp_key) # input in Vdc
        return float(inp_val)
        #return inp_val

    def read_pT(self):
        U=self.read_pT_U()
        T=(U-1.000325)*259.2684
        return T
            
    def reset_T(self):
        self.set_T_U(0.000)

In [ ]:
temp=Temp() # important temperature instance to set temp things manuelly. (One could have included the hardcoded reset commands into measuretemp_curve)
temp.reset_T()
temp.read_pT()

## Temperaturgradient herausfinden
hätte theoretisch benutzt werden können, um den Temp Manager smarter zu machen, also Voltage Erhöhung abhängig von der ermittelten Temperaturdifferenz zu machen und nicht global einstellen zu müsssen


In [ ]:
T=[]
U_set=[]
U_read=[]
U_start=0
stepsize=0.1
U=U_start

while temp.read_pT()<60:
    U=U+stepsize
    U_set.append(U)
    temp.set_T_U(U)
    U_read.append(temp.read_T_U())
    sleep(60)
    T.append(temp.read_pT())
    print("still running")
print("done")

## TempManage Class: Smart Temp Operator


In [14]:
from time import sleep

class TempManage(Temp):

    def __init__(self,step=0.1):
        self.step=step
        Temp.__init__(self)

    def midTemp(self,secs=5):
        temps=np.empty(secs)
        for i in range(secs):
            temps[i]=self.read_pT()
            sleep(1)
        midT=np.mean(temps)
        return midT

    def compareT(self,refT,compT,resolution=1.5): #returns -1, 0, 1 (-1 if compT<refT)
        diff=compT-refT
        if diff <= -resolution:
            return -1
        elif diff <= resolution:
            return 0
        elif diff >= resolution:
            return 1
        else:
            return 0 #add error

    def trenT(self,pastT):
        newT=self.midTemp()
        return self.compareT(pastT,newT), newT

    def restoreStep(self,goalT,T0=None,past_dir=0): #takes stepsize towards goal and returns direction
        if T0 is None:
            T0=self.midTemp()
        U0=self.read_T_U()
        #calculate direction of change
        dir_0=-self.compareT(goalT,T0) #negative direction

        self.smart_stepper(past_dir,dir_0)

        change_U=max(U0+dir_0*self.step,0)
        change_U=min(change_U,1.95)
        self.set_T_U(change_U) #corrects into direction
        return dir_0
            
    def smart_stepper(self,past_dir,dir_0):
        if past_dir==0:
            x=1
        elif past_dir*dir_0==-1:
            x=0.5
        elif past_dir*dir_0==1:
            x=1.05
        else:
            x=0.8
        new_step=min(x*self.step,0.5)
        new_step=max(new_step,0.01)
        self.step=new_step

    def wait_n_check(self): #waits 5s and returns average of later 5s
        sleep(4)
        return self.midTemp()
        
    def adjust_T(self,goalT,T0=None,dir0=0):
        print("adjusting T...")
        if T0 is None:
            T0=self.midTemp()
        newT=T0
        for i in range(100): #to make sure this doesn't run forever
            dir1=self.restoreStep(goalT,past_dir=dir0)
            if dir1==0: #additional check before measurement)
                newT=self.wait_n_check()

                dir0=self.compareT(goalT,newT)
                if dir0==0:
                    break
            else:
                dir0=dir1
            if i % 5==0:
                print("adjuststep:",i," - T=",self.read_pT()," - U=",self.read_T_U())
        return newT
        
#(boss.midTemp())

In [15]:
class TempMeasure(TempManage,Measurement):
    def __init__(self,path,step=0.1):
        TempManage.__init__(self,0.1)
        Measurement.__init__(self,path)

    def measuretemp_curve(self,start,end,vals,dtemp,pathname=None,number=20,log=False): #vals is amount of freqs tested, number is tests for on freq
            if pathname:
                self.pathname=pathname
            self.createtemp_file()
            if log:
                #check that logspace creates enough values
                values=self.get_log_vals(start,end,vals)
            else:
                values=np.linspace(start,end,vals)
            #wait once:
            for freq in values:
                T0=np.array(self.read_pT())
                dir=self.compareT(dtemp,T0)
                if dir!=0:
                    self.adjust_T(dtemp,dir)
                    T0=self.read_pT()
                    print("adjusting done ;)")
                self.set_f(freq)
                self.wait()
                measures=self.stat_measure(number) #arg freq,
                measures=np.concatenate((measures,T0),axis=None)
                self.write_tempData(measures)


# Emails

In [16]:
import smtplib
import time
from email.message import EmailMessage
from datetime import datetime
from tqdm import tqdm
from time import perf_counter
from time import sleep

sender_email = "verflixthuber@gmail.com"
receiver_email = "felix.oberhuber@gmx.at"
app_password = "hdkw kabd cxav imwz"


def send_progress_email(
    subject,
    body,
    sender_email="verflixthuber@gmail.com",
    receiver_email="felix.oberhuber@gmx.at",
    app_password="hdkw kabd cxav imwz"
):
    message = EmailMessage()
    message["From"] = sender_email
    if isinstance(receiver_email,list):
        message["To"] = ", ".join(receiver_email)
    else:
        message["To"] = receiver_email
    message["Subject"] = subject
    message.set_content(body)

    with smtplib.SMTP_SSL("smtp.gmail.com", 465, timeout=30) as smtp:
        smtp.login(sender_email, app_password)
        smtp.send_message(message)

In [23]:
temp.reset_T()
temp.read_pT()

34.398434970000025

## FINAL TEST

In [21]:
empfaenger=["felix.oberhuber@gmx.at","b.k.ballcr@gmail.com"]

start_time=str(datetime.now())
temp.reset_T()

with open(Path.cwd()/"Time"/"Time_spent.txt", "w", encoding="utf-8") as f:
        f.write("Started measurement at: "+start_time)
        f.write("\nThe measurements had a duration of:\n")
temp.reset_T()
ResFreq=0.5*np.array([280.0,1820.0,5120.0])
freq_range=0.5*np.array([120,120,150])
steps=np.array([1,1,1])
for i,freq in enumerate(ResFreq):
    try:
        #send_progress_email("PC2 Measure Update","Part A starts order "+str(i),"verflixthuber@gmail.com",empfaenger,"hdkw kabd cxav imwz")
        start=perf_counter()
        starts,stops=interval_abs(ResFreq[i],freq_range[i])
        name=f"WithoutTemp/final_ResFreq{i}"
        timename=f"ResFreq{i}"
        Messung=Measurement(name)
        Messung.set_f(starts)
        Messung.measure_curve(starts,stops,steps[i]) ## freq=??
        time_spent=(perf_counter()-start)
        current_time=str(datetime.now())
        with open(Path.cwd()/"Time"/"Time_spent.txt", "a", encoding="utf-8") as f:
            f.write(f"{timename}={time_spent:.2f}s")
            f.write("\nfinished this instance at: "+current_time+"\n")
        #send_progress_email("PC2: Measure Update","Part A has done order "+str(i),"verflixthuber@gmail.com",empfaenger,"hdkw kabd cxav imwz")
    except:
        print("Error: PC2: Part A"+str(i)+" crashed")
        #send_progress_email("PC2: Measure Update","Error: PC2: Part A"+str(i)+" crashed","verflixthuber@gmail.com",empfaenger,"hdkw kabd cxav imwz")


with open(Path.cwd()/"Time"/"Time_spent.txt", "a", encoding="utf-8") as f:
    f.write("\nNonTemp measuring done\n")
    f.write("\nStarting Temp Measurement\n")


#send_progress_email("PC2: Measure Update","Part A is done","verflixthuber@gmail.com",empfaenger,"hdkw kabd cxav imwz")

temp.reset_T()
print(temp.read_T_U())
print("Part A done")

##NOW PART B STARTS 

from time import perf_counter
from time import sleep


temp.reset_T() # ganz am Anfang auf 0V setzen
ResFreq=0.5*np.array([280.0])
starts,stops=interval_abs(ResFreq,60)
start_dtemp=np.floor(temp.read_pT())
print("took start_temp",start_dtemp)
try:
    desired_temp=np.arange(start_dtemp,36,2)
    for dtemp in desired_temp:
        start=perf_counter()
        ### templogik zur einstellung oder von dtemp
        # while temp.read_pT()<dtemp:
        set_temp=temp.read_pT()
        #send_progress_email("PC2: Temp Measure Update","Tempmessung startet bei T="+str(dtemp),"verflixthuber@gmail.com",empfaenger,"hdkw kabd cxav imwz")
        name=f"WithTemp/final_TempFreq{dtemp}"
        file_string=f"TempFreq{dtemp}"
        Messung_Temperatur=TempMeasure(name,0.1)  # für temp andere measure curve, die auch noch temperatur speichert?
        Messung_Temperatur.set_f(starts[0])
        Messung_Temperatur.measuretemp_curve(start=starts[0],end=stops[0],vals=3,dtemp=dtemp) #und in measuretemp_curve muss die Temperaturregelung drinne sein sowie das writing
        time_spent=(perf_counter()-start)
        current_time=str(datetime.now())
        with open(Path.cwd()/"Time"/"Time_spent.txt", "a", encoding="utf-8") as f:
            f.write(f"{file_string}={time_spent:.2f}s")
            f.write("\nfinished this instance at: "+current_time+"\n")
        send_progress_email("PC2: Temp Measure Update","Tempmessung fertig von T="+str(dtemp),"verflixthuber@gmail.com",empfaenger,"hdkw kabd cxav imwz")
        
    print("Part B done")
    
except:
    temp.reset_T()
    #send_progress_email("PC2: Temp Measure Update","Error: temp has been reset","verflixthuber@gmail.com",empfaenger,"hdkw kabd cxav imwz")
    
finally:
    temp.reset_T()
temp.reset_T()
current_temp=f"{temp.read_pT():.2f}"
current_voltage=temp.read_T_U()
#send_progress_email("PC2: Temp Measure Update","Temp Measurement done. Temperature has been reset and is:"+current_temp+"\n Voltage reset at "+str(current_voltage),"verflixthuber@gmail.com",empfaenger,"hdkw kabd cxav imwz")
sleep(1)
current_temp=f"{temp.read_pT():.2f}"
current_voltage=temp.read_T_U()
send_progress_email("PC2: Temperature Update","Temperature has been reset and is:"+current_temp+"\n Voltage reset at "+str(current_voltage),"verflixthuber@gmail.com",empfaenger,"hdkw kabd cxav imwz")

100%|██████████| 1/1 [00:03<00:00,  3.68s/it]


0.0
Part A done
took start_temp 34.0
Part B done


# EXPERIMENT LOOP

In [24]:
empfaenger=["felix.oberhuber@gmx.at","b.k.ballcr@gmail.com"]

start_time=str(datetime.now())
temp.reset_T()

with open(Path.cwd()/"Time"/"Time_spent.txt", "w", encoding="utf-8") as f:
        f.write("Started measurement at: "+start_time)
        f.write("\nThe measurements had a duration of:\n")
temp.reset_T()
ResFreq=0.5*np.array([280.0,1820.0,5120.0])
freq_range=0.5*np.array([120,120,150])
steps=np.array([600,600,600])
for i,freq in enumerate(ResFreq):
    try:
        send_progress_email("PC2 Measure Update","Part A starts order "+str(i),"verflixthuber@gmail.com",empfaenger,"hdkw kabd cxav imwz")
        start=perf_counter()
        starts,stops=interval_abs(ResFreq[i],freq_range[i])
        name=f"WithoutTemp/final_ResFreq{i}"
        timename=f"ResFreq{i}"
        Messung=Measurement(name)
        Messung.set_f(starts)
        Messung.measure_curve(starts,stops,steps[i]) ## freq=??
        time_spent=(perf_counter()-start)
        current_time=str(datetime.now())
        with open(Path.cwd()/"Time"/"Time_spent.txt", "a", encoding="utf-8") as f:
            f.write(f"{timename}={time_spent:.2f}s")
            f.write("\nfinished this instance at: "+current_time+"\n")
        send_progress_email("PC2: Measure Update","Part A has done order "+str(i),"verflixthuber@gmail.com",empfaenger,"hdkw kabd cxav imwz")
    except:
        print("Error: PC2: Part A"+str(i)+" crashed")
        send_progress_email("PC2: Measure Update","Error: PC2: Part A"+str(i)+" crashed","verflixthuber@gmail.com",empfaenger,"hdkw kabd cxav imwz")


with open(Path.cwd()/"Time"/"Time_spent.txt", "a", encoding="utf-8") as f:
    f.write("\nNonTemp measuring done\n")
    f.write("\nStarting Temp Measurement\n")


send_progress_email("PC2: Measure Update","Part A is done","verflixthuber@gmail.com",empfaenger,"hdkw kabd cxav imwz")

temp.reset_T()
print(temp.read_T_U())
print("Part A done")

##NOW PART B STARTS 

from time import perf_counter
from time import sleep


temp.reset_T() # ganz am Anfang auf 0V setzen
ResFreq=0.5*np.array([280.0])
starts,stops=interval_abs(ResFreq,60)
start_dtemp=np.floor(temp.read_pT())
print("took start_temp",start_dtemp)

try:
    desired_temp=np.arange(start_dtemp,60,3)
    for dtemp in desired_temp:
        start=perf_counter()
        ### templogik zur einstellung oder von dtemp
        # while temp.read_pT()<dtemp:
        set_temp=temp.read_pT()
        send_progress_email("PC2: Temp Measure Update","Tempmessung startet bei T="+str(dtemp),"verflixthuber@gmail.com",empfaenger,"hdkw kabd cxav imwz")
        name=f"WithTemp/final_TempFreq{dtemp}"
        file_string=f"TempFreq{dtemp}"
        Messung_Temperatur=TempMeasure(name,0.1)  # für temp andere measure curve, die auch noch temperatur speichert?
        Messung_Temperatur.set_f(starts[0])
        Messung_Temperatur.measuretemp_curve(start=starts[0],end=stops[0],vals=600,dtemp=dtemp) #und in measuretemp_curve muss die Temperaturregelung drinne sein sowie das writing
        time_spent=(perf_counter()-start)
        current_time=str(datetime.now())
        with open(Path.cwd()/"Time"/"Time_spent.txt", "a", encoding="utf-8") as f:
            f.write(f"{file_string}={time_spent:.2f}s")
            f.write("\nfinished this instance at: "+current_time+"\n")
        send_progress_email("PC2: Temp Measure Update","Tempmessung fertig von T="+str(dtemp),"verflixthuber@gmail.com",empfaenger,"hdkw kabd cxav imwz")
    print("Part B done")
    
except:
    temp.reset_T()
    send_progress_email("PC2: Temp Measure Update","Error: temp has been reset","verflixthuber@gmail.com",empfaenger,"hdkw kabd cxav imwz")
    
finally:
    temp.reset_T()
temp.reset_T()
current_temp=f"{temp.read_pT():.2f}"
current_voltage=temp.read_T_U()
send_progress_email("PC2: Temp Measure Update","Temp Measurement done. Temperature has been reset (check in 5 min again) and is:"+current_temp+"\n Voltage reset at "+str(current_voltage),"verflixthuber@gmail.com",empfaenger,"hdkw kabd cxav imwz")
sleep(300)
current_temp=f"{temp.read_pT():.2f}"
current_voltage=temp.read_T_U()
send_progress_email("PC2: Temperature Update","Temperature has been reset and is:"+current_temp+"\n Voltage reset at "+str(current_voltage),"verflixthuber@gmail.com",empfaenger,"hdkw kabd cxav imwz")

 43%|████▎     | 257/600 [15:33<20:44,  3.63s/it]/opt/conda/lib/python3.11/site-packages/uncertainties/core.py:1024: UserWarning: Using UFloat objects with std_dev==0 may give unexpected results.
  warn("Using UFloat objects with std_dev==0 may give unexpected results.")
100%|██████████| 600/600 [36:29<00:00,  3.65s/it]


0.0
Part A done
took start_temp 37.0
adjusting T...
adjuststep: 0  - T= 37.50965577000002  - U= 0.1
adjuststep: 5  - T= 38.28746097  - U= 0.68
adjusting done ;)
adjusting T...
adjuststep: 0  - T= 42.69502377000003  - U= 0.843
adjuststep: 5  - T= 43.99136577  - U= 0.19
adjuststep: 10  - T= 43.99136577  - U= 0.0
adjuststep: 15  - T= 43.47282897  - U= 0.0
adjuststep: 20  - T= 42.69502377000003  - U= 0.0
adjuststep: 25  - T= 42.17648697000003  - U= 0.0
adjusting done ;)
adjusting T...
adjusting done ;)
adjusting T...
adjuststep: 0  - T= 38.28746097  - U= 0.0
adjusting done ;)
adjusting T...
adjuststep: 0  - T= 42.69502377000003  - U= 0.32
adjuststep: 5  - T= 42.69502377000003  - U= 0.0
adjuststep: 10  - T= 42.17648697000003  - U= 0.0
adjusting done ;)
adjusting T...
adjusting done ;)
adjusting T...
adjuststep: 0  - T= 38.80599777  - U= 0.135
adjusting done ;)
adjusting T...
adjuststep: 0  - T= 42.17648697000003  - U= 0.025
adjusting done ;)
adjusting T...
adjuststep: 0  - T= 40.10233977000